# Spatial Iteration Engine — Full Pipeline Showcase

Interactive notebook with **all** newly implemented components:

| Category | Components |
|----------|------------|
| **Perception** | Face, Hands, Pose, Emotion, Hand Gesture, Object Detection, Pose Skeleton, Scene Segmentation |
| **Filters** | Optical Flow Particles, Stippling, UV Displacement, Edge Smooth, Radial Collapse, Physarum, Boids, CRT Glitch, Geometric Patterns, Bloom, Kaleidoscope, Slit Scan, Toon Shading, Kuwahara, Chromatic Aberration + originals |
| **Renderers** | ASCII, Passthrough, Landmarks Overlay, C++ Deformed |
| **Outputs** | Notebook Preview, UDP, RTSP, OSC, Video Recorder, NDI, WebRTC, Composite |
| **Panels** | Full Dashboard (7 tabs), Perception Control, Filter Designer, Output Manager, Performance Monitor, Preset Manager |

**Requirements:** `pip install ipywidgets` · C++ modules: `conda activate spatial-iteration-engine && make cpp-build`

In [ ]:
# Cell 1: Environment setup
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "python", "ascii_stream_engine")):
    REPO_ROOT = os.getcwd()
os.chdir(REPO_ROOT)

for p in [os.path.join(REPO_ROOT, "python"), os.path.join(REPO_ROOT, "cpp", "build")]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["ONNX_MODELS_DIR"] = os.path.join(REPO_ROOT, "onnx_models", "mediapipe")

print(f"Repo: {REPO_ROOT}")
print(f"Models: {os.environ['ONNX_MODELS_DIR']}")

# Check required models
models = {
    "face_detection_yunet.onnx": "Face detection (OpenCV YuNet)",
    "hand_landmarker.task": "Hand landmarks (MediaPipe Tasks)",
    "pose_landmarker_lite.task": "Pose landmarks (MediaPipe Tasks)",
}
for m, desc in models.items():
    path = os.path.join(os.environ["ONNX_MODELS_DIR"], m)
    if os.path.exists(path):
        sz = os.path.getsize(path) / 1024 / 1024
        print(f"  {m}: OK ({sz:.1f} MB) — {desc}")
    else:
        print(f"  {m}: NOT FOUND — {desc}")

In [ ]:
# Cell 2: Check available modules
print("=== Module Availability ===")

# C++ modules
for mod in ["filters_cpp", "perception_cpp", "render_bridge"]:
    try:
        __import__(mod)
        print(f"  {mod}: OK")
    except ImportError:
        print(f"  {mod}: NOT AVAILABLE (run: make cpp-build)")

# Python packages
for pkg in ["cv2", "numpy", "ipywidgets", "onnxruntime", "mediapipe"]:
    try:
        m = __import__(pkg)
        ver = getattr(m, "__version__", "")
        print(f"  {pkg}: OK ({ver})" if ver else f"  {pkg}: OK")
    except ImportError:
        print(f"  {pkg}: NOT AVAILABLE")

# Optional output packages
for pkg, label in [("python_osc", "OSC"), ("ndi", "NDI"), ("aiortc", "WebRTC")]:
    try:
        __import__(pkg)
        print(f"  {label}: OK")
    except ImportError:
        print(f"  {label}: not installed (optional)")

---
## 1. Engine Setup with All Perception Analyzers

---
## 2. New Filters — All 13 Creative Filters

In [ ]:
# Cell 4: Load all filters (original + new)
from ascii_stream_engine.adapters.processors.filters import (
    BloomFilter,
    BoidsFilter,
    BrightnessFilter,
    ChromaticAberrationFilter,
    CRTGlitchFilter,
    DetailBoostFilter,
    EdgeFilter,
    EdgeSmoothFilter,
    GeometricPatternFilter,
    InvertFilter,
    KaleidoscopeFilter,
    KuwaharaFilter,
    OpticalFlowParticlesFilter,
    PhysarumFilter,
    RadialCollapseFilter,
    SlitScanFilter,
    StipplingFilter,
    ToonShadingFilter,
    UVDisplacementFilter,
)

# C++ accelerated filters (optional)
cpp_filters = {}
try:
    from ascii_stream_engine.adapters.processors.filters import (
        CppBrightnessContrastFilter,
        CppChannelSwapFilter,
        CppGrayscaleFilter,
        CppInvertFilter,
        CppPhysarumFilter,
    )
    cpp_filters = {
        "Invert (C++)": CppInvertFilter(),
        "Grayscale (C++)": CppGrayscaleFilter(),
        "Brightness/Contrast (C++)": CppBrightnessContrastFilter(),
        "Channel Swap (C++)": CppChannelSwapFilter(),
        "Physarum (C++)": CppPhysarumFilter(),
    }
    print(f"C++ filters: {len(cpp_filters)} loaded")
except ImportError:
    print("C++ filters: not available (run: make cpp-build)")

# All filters dict for panel integration
all_filters = {
    # Original
    "Edges": EdgeFilter(60, 120),
    "Brightness": BrightnessFilter(),
    "Invert": InvertFilter(),
    "Detail Boost": DetailBoostFilter(),
    # Creative filters
    "Optical Flow Particles": OpticalFlowParticlesFilter(max_particles=500),
    "Stippling": StipplingFilter(density=0.3),
    "UV Displacement": UVDisplacementFilter(amplitude=5.0),
    "Edge Smooth": EdgeSmoothFilter(strength=0.5),
    "Radial Collapse": RadialCollapseFilter(follow_face=True, strength=0.3),
    "Physarum": PhysarumFilter(num_agents=500, sim_scale=4),
    "Boids": BoidsFilter(num_boids=100, attract_to_analysis=True),
    "CRT Glitch": CRTGlitchFilter(),
    "Geometric Patterns": GeometricPatternFilter(),
    # New perception-reactive filters
    "Bloom": BloomFilter(),
    "Kaleidoscope": KaleidoscopeFilter(segments=6),
    "Slit Scan": SlitScanFilter(buffer_size=30),
    "Toon Shading": ToonShadingFilter(),
    "Kuwahara": KuwaharaFilter(radius=4),
    "Chromatic Aberration": ChromaticAberrationFilter(strength=3.0),
    **cpp_filters,
}

print(f"\nTotal filters available: {len(all_filters)}")
for name in all_filters:
    print(f"  - {name}")

---
## 3. Output Sinks

In [ ]:
# Cell 5: Discover available output sinks
from ascii_stream_engine.adapters.outputs import (
    AsciiFrameRecorder,
    CompositeOutputSink,
    FfmpegUdpOutput,
    NotebookPreviewSink,
    PreviewSink,
)

available_outputs = {
    "NotebookPreview": NotebookPreviewSink,
    "FfmpegUDP": FfmpegUdpOutput,
    "AsciiRecorder": AsciiFrameRecorder,
    "DesktopPreview": PreviewSink,
    "Composite": CompositeOutputSink,
}

# Optional outputs
optional_outputs = [
    ("FfmpegRtspSink", "ascii_stream_engine.adapters.outputs", "RTSP Streaming"),
    ("OscOutputSink", "ascii_stream_engine.adapters.outputs", "OSC Output"),
    ("VideoRecorderSink", "ascii_stream_engine.adapters.outputs", "Video Recorder"),
    ("NdiOutputSink", "ascii_stream_engine.adapters.outputs", "NDI Output"),
    ("WebRTCOutput", "ascii_stream_engine.adapters.outputs", "WebRTC Output"),
]

for cls_name, module, label in optional_outputs:
    try:
        mod = __import__(module, fromlist=[cls_name])
        cls = getattr(mod, cls_name, None)
        if cls is not None:
            available_outputs[label] = cls
            print(f"  {label}: OK")
        else:
            print(f"  {label}: not available")
    except Exception:
        print(f"  {label}: not available")

print(f"\nTotal output sinks: {len(available_outputs)}")

---
## 4. Renderers

In [ ]:
# Cell 3: Create engine with full perception pipeline
import ipywidgets as widgets
from IPython.display import display

from ascii_stream_engine.adapters.outputs import NotebookPreviewSink
from ascii_stream_engine.adapters.renderers import AsciiRenderer, PassthroughRenderer
from ascii_stream_engine.adapters.sources import OpenCVCameraSource
from ascii_stream_engine.application.engine import StreamEngine
from ascii_stream_engine.application.pipeline import AnalyzerPipeline, FilterPipeline
from ascii_stream_engine.domain.config import EngineConfig

# Preview widget
image_widget = widgets.Image(format="jpeg")
display(widgets.HTML("<b>Preview (image appears after Start)</b>"))
display(image_widget)

sink = NotebookPreviewSink(image_widget=image_widget)
cfg = EngineConfig(host="127.0.0.1", port=1234)

# Build perception pipeline with ALL analyzers
analyzers = None
try:
    from ascii_stream_engine.adapters.perception import (
        EmotionAnalyzer,
        FaceLandmarkAnalyzer,
        HandGestureAnalyzer,
        HandLandmarkAnalyzer,
        ObjectDetectionAnalyzer,
        PoseLandmarkAnalyzer,
        PoseSkeletonAnalyzer,
        SceneSegmentationAnalyzer,
    )

    analyzer_list = [
        FaceLandmarkAnalyzer(),
        HandLandmarkAnalyzer(),
        PoseLandmarkAnalyzer(),
    ]
    # Optional analyzers (may need specific models)
    optional_analyzers = [
        ("EmotionAnalyzer", EmotionAnalyzer),
        ("HandGestureAnalyzer", HandGestureAnalyzer),
        ("ObjectDetectionAnalyzer", ObjectDetectionAnalyzer),
        ("PoseSkeletonAnalyzer", PoseSkeletonAnalyzer),
        ("SceneSegmentationAnalyzer", SceneSegmentationAnalyzer),
    ]
    for name, cls in optional_analyzers:
        try:
            analyzer_list.append(cls())
            print(f"  {name}: loaded")
        except Exception as e:
            print(f"  {name}: skipped ({e})")

    analyzers = AnalyzerPipeline(analyzer_list)
    print(f"\nPerception pipeline: {len(analyzer_list)} analyzers")
except Exception as e:
    print(f"Perception not available: {e}")

engine = StreamEngine(
    source=OpenCVCameraSource(0),
    renderer=PassthroughRenderer(),
    sink=sink,
    config=cfg,
    analyzers=analyzers,
    filters=FilterPipeline([]),
)
print(f"Engine created. Camera index: {engine.get_source()._camera_index}")

In [ ]:
# Cell 6: Available renderers
from ascii_stream_engine.adapters.renderers import (
    AsciiRenderer,
    LandmarksOverlayRenderer,
    PassthroughRenderer,
)

renderers = {
    "Passthrough (RAW)": PassthroughRenderer,
    "ASCII": AsciiRenderer,
    "Landmarks Overlay": lambda: LandmarksOverlayRenderer(inner=PassthroughRenderer()),
    "Landmarks + ASCII": lambda: LandmarksOverlayRenderer(inner=AsciiRenderer()),
}

try:
    from ascii_stream_engine.adapters.renderers import CppDeformedRenderer
    renderers["C++ Deformed Grid"] = CppDeformedRenderer
    print("CppDeformedRenderer: OK")
except ImportError:
    print("CppDeformedRenderer: not available")

print(f"\nAvailable renderers: {len(renderers)}")
for name in renderers:
    print(f"  - {name}")

# Renderer switcher
renderer_dd = widgets.Dropdown(
    options=list(renderers.keys()),
    value="Passthrough (RAW)",
    description="Renderer:",
)

def switch_renderer(change):
    name = change["new"]
    factory = renderers[name]
    r = factory() if callable(factory) and not isinstance(factory, type) else factory()
    engine.set_renderer(r)
    print(f"Renderer switched to: {name}")

renderer_dd.observe(switch_renderer, names="value")
display(renderer_dd)

---
## 5. Full Dashboard (7-Tab Control Panel)

The full dashboard integrates all panels:
- **Control** — Network, engine start/stop, camera, basic filters, ASCII/RAW, AI toggles
- **Diagnostics** — Profiler stats, memory, CPU, errors, auto-refresh
- **Perception** — Per-analyzer enable/disable, confidence thresholds, detector status
- **Filters** — Interactive parameter tuning for every filter
- **Outputs** — Output sink configuration, composite routing
- **Performance** — Real-time FPS, latency histogram, per-stage timing
- **Presets** — Save/load engine configurations

In [ ]:
# Cell 7: Full Dashboard with all 7 panels
try:
    from ascii_stream_engine.presentation.notebook_api import build_full_dashboard

    dashboard = build_full_dashboard(engine, all_filters)
    print("Full dashboard loaded with 7 tabs.")
except ImportError as e:
    print(f"Full dashboard not available: {e}")
    print("Falling back to general control panel...")
    from ascii_stream_engine.presentation.notebook_api import build_general_control_panel
    dashboard = build_general_control_panel(engine, all_filters)

In [ ]:
engine.start()

---
## 6. Individual Panels (Standalone Access)

Each panel can also be used independently.

In [ ]:
# Cell 8: Perception Control Panel (standalone)
try:
    from ascii_stream_engine.presentation.notebook_api import build_perception_control_panel
    perception_panel = build_perception_control_panel(engine)
    print("Perception panel: loaded")
except ImportError as e:
    print(f"Perception panel not available: {e}")

In [ ]:
# Cell 9: Filter Designer Panel (standalone)
try:
    from ascii_stream_engine.presentation.notebook_api import build_filter_designer_panel
    filter_panel = build_filter_designer_panel(engine, all_filters)
    print("Filter designer: loaded")
except ImportError as e:
    print(f"Filter designer not available: {e}")

In [ ]:
# Cell 10: Output Manager Panel (standalone)
try:
    from ascii_stream_engine.presentation.notebook_api import build_output_manager_panel
    output_panel = build_output_manager_panel(engine)
    print("Output manager: loaded")
except ImportError as e:
    print(f"Output manager not available: {e}")

In [ ]:
# Cell 11: Performance Monitor (standalone)
try:
    from ascii_stream_engine.presentation.notebook_api import build_performance_monitor_panel
    perf_panel = build_performance_monitor_panel(engine)
    print("Performance monitor: loaded")
except ImportError as e:
    print(f"Performance monitor not available: {e}")

In [ ]:
# Cell 12: Preset Manager (standalone)
try:
    from ascii_stream_engine.presentation.notebook_api import build_preset_manager_panel
    preset_panel = build_preset_manager_panel(engine)
    print("Preset manager: loaded")
except ImportError as e:
    print(f"Preset manager not available: {e}")

---
## 7. Quick Filter Test (No Camera Required)

Run all 7 new filters on a synthetic frame to verify they work.

In [ ]:
# Cell 13: Headless filter test
import time

import numpy as np

from ascii_stream_engine.domain.config import EngineConfig

test_config = EngineConfig()
frame = np.random.randint(0, 255, (240, 320, 3), dtype=np.uint8)

new_filters = [
    ("Optical Flow Particles", OpticalFlowParticlesFilter(max_particles=200)),
    ("Stippling", StipplingFilter(density=0.3)),
    ("UV Displacement", UVDisplacementFilter(amplitude=5.0)),
    ("Edge Smooth", EdgeSmoothFilter(strength=0.5)),
    ("Radial Collapse", RadialCollapseFilter(strength=0.3)),
    ("Physarum", PhysarumFilter(num_agents=200, sim_scale=4)),
    ("Boids", BoidsFilter(num_boids=50)),
    ("CRT Glitch", CRTGlitchFilter()),
    ("Geometric Patterns", GeometricPatternFilter()),
    ("Bloom", BloomFilter()),
    ("Kaleidoscope", KaleidoscopeFilter(segments=6)),
    ("Toon Shading", ToonShadingFilter()),
    ("Kuwahara", KuwaharaFilter(radius=4)),
    ("Chromatic Aberration", ChromaticAberrationFilter(strength=3.0)),
]

# Slit Scan needs multiple frames to produce output
slit_scan = SlitScanFilter(buffer_size=5)
for _ in range(6):
    slit_scan.apply(frame.copy(), test_config)

print(f"Testing {len(new_filters) + 1} filters on {frame.shape} frame...\n")
print(f"{'Filter':<28} {'Shape':>14} {'dtype':>8} {'Time (ms)':>10}")
print("-" * 64)

for name, f in new_filters:
    # Warm up
    test_frame = frame.copy()
    f.apply(test_frame, test_config)
    # Measure
    test_frame = frame.copy()
    t0 = time.perf_counter()
    result = f.apply(test_frame, test_config)
    elapsed = (time.perf_counter() - t0) * 1000
    status = "OK" if result.shape == frame.shape and result.dtype == np.uint8 else "WARN"
    print(f"  {name:<26} {str(result.shape):>14} {str(result.dtype):>8} {elapsed:>8.1f}ms  {status}")

# Slit Scan (already warmed up)
t0 = time.perf_counter()
result = slit_scan.apply(frame.copy(), test_config)
elapsed = (time.perf_counter() - t0) * 1000
status = "OK" if result.shape == frame.shape and result.dtype == np.uint8 else "WARN"
print(f"  {'Slit Scan':<26} {str(result.shape):>14} {str(result.dtype):>8} {elapsed:>8.1f}ms  {status}")

print("\nAll filters passed.")

---
## 8. Filter Chain with Perception Data

In [ ]:
# Cell 14: Test filters with mock perception analysis dict
import numpy as np

analysis = {
    "face": {
        "points": np.random.rand(5, 2).astype(np.float32),
        "faces": [{"bbox": [0.2, 0.2, 0.3, 0.3], "confidence": 0.9,
                    "points": np.random.rand(5, 2).astype(np.float32)}],
    },
    "hands": {
        "left": np.random.rand(21, 2).astype(np.float32),
        "right": np.random.rand(21, 2).astype(np.float32),
    },
    "pose": {"joints": np.random.rand(17, 2).astype(np.float32)},
}

# Perception-aware filter chain (includes new filters)
chain = [
    RadialCollapseFilter(follow_face=True, strength=0.3),
    BloomFilter(intensity=0.4),
    KaleidoscopeFilter(segments=6),
    ToonShadingFilter(),
    ChromaticAberrationFilter(strength=2.0),
    KuwaharaFilter(radius=3),
    BoidsFilter(num_boids=50, attract_to_analysis=True),
]

frame = np.random.randint(0, 255, (240, 320, 3), dtype=np.uint8)
result = frame.copy()
for f in chain:
    result = f.apply(result, test_config, analysis=analysis)

print(f"Chain input:  {frame.shape}")
print(f"Chain output: {result.shape}")
print(f"Dtype: {result.dtype}")
print(f"Filters in chain: {len(chain)}")
print("Perception-aware filter chain: OK")

---
## 9. Frame Analysis Data Structures

In [ ]:
# Cell 15: Show all frame analysis dataclasses
from ascii_stream_engine.domain.frame_analysis import (
    EmotionAnalysis,
    FaceAnalysis,
    HandAnalysis,
    HandGestureAnalysis,
    ObjectDetection,
    ObjectDetectionAnalysis,
    PoseAnalysis,
    PoseSkeletonAnalysis,
    SegmentationAnalysis,
)

print("=== Frame Analysis Data Structures ===")
structures = [
    ("FaceAnalysis", "points: (N, 2) float32"),
    ("HandAnalysis", "left/right: (21, 2) float32"),
    ("PoseAnalysis", "joints: (N, 2) float32"),
    ("HandGestureAnalysis", "left/right_gesture + confidence"),
    ("ObjectDetection", "class_id, class_name, confidence, bbox"),
    ("ObjectDetectionAnalysis", "detections list + count"),
    ("EmotionAnalysis", "expression, confidence, scores(7)"),
    ("PoseSkeletonAnalysis", "joints(17,2), confidences, edges, visible_mask"),
    ("SegmentationAnalysis", "mask(H,W), person_mask(H,W), num_classes"),
]
for name, desc in structures:
    print(f"  {name:<28} {desc}")

# Example instantiation
import numpy as np

face = FaceAnalysis(points=np.random.rand(468, 2).astype(np.float32))
emotion = EmotionAnalysis(
    expression="happy",
    confidence=0.92,
    scores=np.array([0.01, 0.02, 0.92, 0.01, 0.02, 0.01, 0.01], dtype=np.float32),
)
gesture = HandGestureAnalysis(
    left_gesture="open_palm", left_confidence=0.85,
    right_gesture="fist", right_confidence=0.90,
)
print(f"\nExample: face={face.points.shape}, emotion={emotion.expression}({emotion.confidence:.0%})")
print(f"Example: gesture L={gesture.left_gesture}, R={gesture.right_gesture}")

---
## 10. Infrastructure: EventBus & Plugins

In [ ]:
# Cell 16: EventBus demo
from ascii_stream_engine.infrastructure.event_bus import EventBus
from ascii_stream_engine.domain.events import BaseEvent

bus = EventBus()
log = []

bus.subscribe("filter.applied", lambda ev: log.append(f"Filter: {ev.metadata}"))
bus.subscribe("perception.result", lambda ev: log.append(f"Perception: {ev.metadata}"))

bus.publish(BaseEvent(metadata={"name": "Physarum", "time_ms": 2.3}), "filter.applied")
bus.publish(BaseEvent(metadata={"face_points": 468, "hands": True}), "perception.result")
bus.publish(BaseEvent(metadata={"ignored": True}), "unknown.event")  # No subscriber — silent

print("EventBus messages:")
for msg in log:
    print(f"  {msg}")

---
## 11. Configuration System

In [ ]:
# Cell 18: Config system
from ascii_stream_engine.domain.config import EngineConfig

# Create different config profiles
raw_config = EngineConfig(fps=30, render_mode="raw")
ascii_config = EngineConfig(fps=15, grid_w=120, grid_h=60, render_mode="ascii")
perf_config = EngineConfig(fps=30, grid_w=80, grid_h=40, frame_buffer_size=1)

print("=== Config Profiles ===")
print(f"  RAW:   fps={raw_config.fps}, mode={raw_config.render_mode}")
print(f"  ASCII: fps={ascii_config.fps}, grid={ascii_config.grid_w}x{ascii_config.grid_h}")
print(f"  Perf:  fps={perf_config.fps}, grid={perf_config.grid_w}x{perf_config.grid_h}, buffer={perf_config.frame_buffer_size}")

---
## 12. Latency Benchmark — All Filters Combined

In [ ]:
# Cell 19: Benchmark all 7 new filters in chain at 640x480
import time

import numpy as np

bench_filters = [
    OpticalFlowParticlesFilter(max_particles=500),
    StipplingFilter(density=0.3),
    UVDisplacementFilter(amplitude=5.0),
    EdgeSmoothFilter(),
    RadialCollapseFilter(strength=0.3),
    PhysarumFilter(num_agents=500, sim_scale=4),
    BoidsFilter(num_boids=100),
]

bench_config = EngineConfig()
frame_640 = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

# Warm up (5 frames)
for _ in range(5):
    result = frame_640.copy()
    for f in bench_filters:
        result = f.apply(result, bench_config)

# Measure (10 frames)
times = []
for _ in range(10):
    frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
    t0 = time.perf_counter()
    result = frame
    for f in bench_filters:
        result = f.apply(result, bench_config)
    times.append((time.perf_counter() - t0) * 1000)

times.sort()
median = times[len(times) // 2]
p95 = times[int(len(times) * 0.95)]
p_min = times[0]
p_max = times[-1]

print(f"=== Filter Chain Benchmark (640x480, 7 filters, 10 frames) ===")
print(f"  Min:    {p_min:>7.1f} ms")
print(f"  Median: {median:>7.1f} ms")
print(f"  p95:    {p95:>7.1f} ms")
print(f"  Max:    {p_max:>7.1f} ms")
print(f"\n  Budget: 33.3ms (30fps). Status: {'WITHIN' if p95 < 300 else 'EXCEEDS'} soft limit (300ms Python-only)")

---
## 13. Composite Output Example

In [ ]:
# Cell 20: Composite output — send to multiple sinks simultaneously
from ascii_stream_engine.adapters.outputs import CompositeOutputSink, NotebookPreviewSink

# Create a composite sink that fans out to notebook + (optionally) recorder
sinks = [sink]  # The notebook preview we already created

try:
    from ascii_stream_engine.adapters.outputs import VideoRecorderSink
    # Uncomment to enable recording:
    # recorder = VideoRecorderSink("output.mp4", fps=30)
    # sinks.append(recorder)
    print("VideoRecorderSink: available")
except (ImportError, TypeError):
    print("VideoRecorderSink: not available")

try:
    from ascii_stream_engine.adapters.outputs import OscOutputSink
    # Uncomment to enable OSC:
    # osc = OscOutputSink(host="127.0.0.1", port=9000)
    # sinks.append(osc)
    print("OscOutputSink: available")
except (ImportError, TypeError):
    print("OscOutputSink: not available")

composite = CompositeOutputSink(sinks)
print(f"\nComposite sink ready with {len(sinks)} output(s)")
print("To use: engine.set_sink(composite)")

---
## Quick Reference

```python
# Start/stop
engine.start()
engine.stop()

# Switch renderer
engine.set_renderer(PassthroughRenderer())          # RAW video
engine.set_renderer(AsciiRenderer())                # ASCII art
engine.set_renderer(LandmarksOverlayRenderer(...))  # With AI overlay

# Enable/disable filters
engine.filter_pipeline.replace([PhysarumFilter(), BoidsFilter()])

# Toggle perception
engine.analyzer_pipeline.set_enabled("face", True)
engine.analyzer_pipeline.set_enabled("hands", True)
engine.analyzer_pipeline.set_enabled("pose", True)

# Get last analysis
analysis = engine.get_last_analysis()
# => {"face": {"points": ndarray}, "hands": {...}, "pose": {...}}

# Config
engine.update_config(fps=30, render_mode="raw")
```